# Phase 2 — Temporal Leakage & Early-Warning Audit V1
**Read-only, independent evaluation.** Reuses saved prediction files and 34-feature dataset. Does not retrain or change labels/thresholds.

**Interpretation:** Static checks and checkpoint-aligned prediction diagnostics are not proof of absence of future leakage. True event lead time is reported only when independently dated outcome events are available.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; adjust ROOT manually.')
ROOT=Path('/content/drive/MyDrive/GC1/Temporal_snapshots')
FEAT=ROOT/'feature_engineering_output'/'tabpfn_labelled_dataset.parquet'
PRED=ROOT/'model_training_efficient_investigation'/'predictions'
SPLIT=ROOT/'model_training_output'/'project_split_manifest.csv'
SELECTED=ROOT/'feature_engineering_output'/'selected_feature_list.csv'
OUT=Path('/content/drive/MyDrive/02_Risk_Prediction/Results/Temporal_Leakage_Early_Warning_V1')
OUT.mkdir(parents=True,exist_ok=True)
TARGETS=['schedule_risk_label','scope_risk_label','dependency_risk_label']
KEYS=['instance_id','week_number','checkpoint_time']
FILES={'features':FEAT,'validation':PRED/'validation_predictions.parquet','test':PRED/'exploratory_test_predictions.parquet','split':SPLIT,'selected_features':SELECTED}
missing={k:str(p) for k,p in FILES.items() if not p.exists()}
if missing: raise FileNotFoundError(json.dumps(missing,indent=2))
print('Inputs present. Results:',OUT)

Mounted at /content/drive
Inputs present. Results: /content/drive/MyDrive/02_Risk_Prediction/Results/Temporal_Leakage_Early_Warning_V1


## 1. Verify checkpoint, partition and prediction alignment

In [4]:
print(df[['partition', 'evaluation_partition']].value_counts(dropna=False))

partition   evaluation_partition
validation  validation              594
test        test                    591
Name: count, dtype: int64


In [5]:
print("Partition values:", df['partition'].unique())
print("Evaluation partition values:", df['evaluation_partition'].unique())

Partition values: ['validation' 'test']
Evaluation partition values: ['validation' 'test']


In [11]:
features = pd.read_parquet(FEAT)
split = pd.read_csv(SPLIT)

val = pd.read_parquet(FILES['validation']).assign(
    evaluation_partition='validation'
)
test = pd.read_parquet(FILES['test']).assign(
    evaluation_partition='test'
)

preds = pd.concat([val, test], ignore_index=True)

for df in [features, preds]:
    df['instance_id'] = df['instance_id'].astype(str)
    df['week_number'] = pd.to_numeric(
        df['week_number'], errors='raise'
    ).astype(int)
    df['checkpoint_time'] = pd.to_datetime(
        df['checkpoint_time'],
        utc=True,
        format='mixed',
        errors='coerce'
    )
    assert df['checkpoint_time'].notna().all()

split['instance_id'] = split['instance_id'].astype(str)
split['partition'] = (
    split['partition']
    .astype(str)
    .str.lower()
    .replace({
        'val': 'validation',
        'exploratory_test': 'test'
    })
)

# Original validation checks
assert not features.duplicated(KEYS).any()
assert not preds.duplicated(KEYS + ['risk']).any()
assert set(preds['risk']) == set(TARGETS)
assert split['instance_id'].is_unique
assert set(split['partition']) == {'train', 'validation', 'test'}
assert set(features['instance_id']) == set(split['instance_id'])
assert set(preds['instance_id']).issubset(
    set(split['instance_id'])
)

assert preds['probability'].between(0, 1).all()
assert preds['threshold'].between(0, 1).all()

assert preds['predicted_label'].eq(
    preds['probability'].ge(preds['threshold']).astype(int)
).all()

# FIX: Give the split-manifest partition a distinct name
split_check = split[
    ['instance_id', 'partition']
].rename(columns={
    'partition': 'manifest_partition'
})

joined = preds.merge(
    split_check,
    on='instance_id',
    validate='many_to_one'
)

# Validate all three partition indicators
assert joined['partition'].eq(
    joined['manifest_partition']
).all(), 'Prediction/manifest partition mismatch'

assert joined['evaluation_partition'].eq(
    joined['manifest_partition']
).all(), 'Evaluation/manifest partition mismatch'

# Verify labels against original feature dataset
truth = features[KEYS + TARGETS].melt(
    id_vars=KEYS,
    var_name='risk',
    value_name='source_label'
)

joined = joined.merge(
    truth,
    on=KEYS + ['risk'],
    validate='one_to_one',
    indicator=True
)

assert joined['_merge'].eq('both').all(), (
    'Prediction keys missing from ground truth'
)

assert joined['actual_label'].eq(
    joined['source_label']
).all(), 'Prediction labels mismatch ground truth'

print('All partition and label checks passed!')
print(
    'Predictions:', len(joined),
    'Checkpoints:',
    joined[KEYS].drop_duplicates().shape[0]
)

display(
    joined.groupby(['partition', 'risk'])
    .size()
    .unstack(fill_value=0)
)

All partition and label checks passed!
Predictions: 1185 Checkpoints: 395


risk,dependency_risk_label,schedule_risk_label,scope_risk_label
partition,,,
test,197,197,197
validation,198,198,198


## 2. Feature-column and time metadata leakage screen
This flags suspicious names and timestamps. It cannot prove how upstream feature values were calculated. The review covers source code of snapshot and feature generation before claiming full leakage safety.

In [12]:
sel=pd.read_csv(SELECTED)
print('Selected feature list columns:',list(sel.columns))
print(sel.head(5).to_string(index=False))
# Find feature-name column without silently assuming a schema.
name_candidates=[c for c in sel.columns if 'feature' in c.lower() or 'column' in c.lower()]
if len(name_candidates)==1:
    feature_names=sel[name_candidates[0]].dropna().astype(str).tolist()
elif sel.shape[1]==1:
    feature_names=sel.iloc[:,0].dropna().astype(str).tolist()
else:
    feature_names=[]
    print('WARNING: Ambiguous selected-feature CSV schema; inspect manually.')
patterns=r'(^|_)(future|next|outcome|target|label|lead|horizon|post|ground_truth|failure_date)(_|$)'
scan=pd.DataFrame([{'feature':f,'suspicious_name':bool(re.search(patterns,f,re.I)),'present_in_dataset':f in features.columns} for f in feature_names])
scan.to_csv(OUT/'feature_name_screen.csv',index=False)
display(scan[scan['suspicious_name']] if len(scan) else scan)
if feature_names:
    print('Selected features:',len(feature_names),'Missing:',int((~scan['present_in_dataset']).sum()),'Flagged:',int(scan['suspicious_name'].sum()))
# Timestamp-bearing columns may be legitimate metadata; inspect availability semantics manually.
time_cols=[c for c in features.columns if re.search(r'(date|time|observed|available|as_of)',c,re.I)]
print('Timestamp-like columns:',time_cols)
print('IMPORTANT: This is a screen, not a proof of no leakage.')

Selected feature list columns: ['feature_name']
         feature_name
         total_issues
     open_issue_ratio
      reopened_issues
blocked_status_issues
 unknown_status_ratio


,feature,suspicious_name,present_in_dataset


Selected features: 34 Missing: 0 Flagged: 0
Timestamp-like columns: ['checkpoint_time']
IMPORTANT: This is a screen, not a proof of no leakage.


## 3. Chronological sanity checks
Checks per-instance checkpoint ordering, uniqueness and suspicious timestamp fields. These do not reconstruct the graph at each checkpoint.

In [13]:
chron=[]
for inst,g in features.sort_values('checkpoint_time').groupby('instance_id'):
    g=g.sort_values('week_number')
    chron.append({'instance_id':inst,'rows':len(g),'duplicate_weeks':int(g['week_number'].duplicated().sum()),'non_increasing_checkpoint_times':int((g['checkpoint_time'].diff().dropna()<=pd.Timedelta(0)).sum()),'non_increasing_week_numbers':int((g['week_number'].diff().dropna()<=0).sum())})
chron=pd.DataFrame(chron)
chron.to_csv(OUT/'checkpoint_chronology.csv',index=False)
display(chron.describe(include='all'))
print('Chronology violations:',int(chron[['duplicate_weeks','non_increasing_checkpoint_times','non_increasing_week_numbers']].sum().sum()))

,instance_id,rows,duplicate_weeks,non_increasing_checkpoint_times,non_increasing_week_numbers
count,30,30.000000,30.0,30.0,30.0
unique,30,NaN,NaN,NaN,NaN
top,AEROGEAR-I01,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,32.900000,0.0,0.0,0.0
std,NaN,0.803012,0.0,0.0,0.0
min,NaN,29.000000,0.0,0.0,0.0
25%,NaN,33.000000,0.0,0.0,0.0
50%,NaN,33.000000,0.0,0.0,0.0
75%,NaN,33.000000,0.0,0.0,0.0


Chronology violations: 0


## 4. Early-warning diagnostics based on the 14-day labels
A true positive warning here means the model predicted a positive **14-day outcome label** at a checkpoint. This is **not** the number of days before the underlying event. We report alert recall, precision and workload without inventing event onset dates.

In [14]:
rows=[]
for (part,risk),g in joined.groupby(['partition','risk']):
    y=g['actual_label'].astype(int); p=g['predicted_label'].astype(int)
    tp=int(((y==1)&(p==1)).sum());fp=int(((y==0)&(p==1)).sum());fn=int(((y==1)&(p==0)).sum());tn=int(((y==0)&(p==0)).sum())
    rows.append({'partition':part,'risk':risk,'checkpoints':len(g),'positives':int(y.sum()),'alerts':int(p.sum()),'tp':tp,'fp':fp,'fn':fn,'tn':tn,'positive_alert_precision':tp/(tp+fp) if tp+fp else np.nan,'positive_outcome_recall':tp/(tp+fn) if tp+fn else np.nan,'false_alerts_per_100_checkpoints':100*fp/len(g),'alert_rate':float(p.mean())})
warning=pd.DataFrame(rows)
warning.to_csv(OUT/'early_warning_checkpoint_metrics.csv',index=False)
display(warning)
print('Actual lead-time days: NOT ESTABLISHED — need independent event-onset timestamps and a documented episode definition.')

,partition,risk,checkpoints,positives,alerts,tp,fp,fn,tn,positive_alert_precision,positive_outcome_recall,false_alerts_per_100_checkpoints,alert_rate
0,test,dependency_risk_label,197,14,59,9,50,5,133,0.152542,0.642857,25.380711,0.299492
1,test,schedule_risk_label,197,52,95,46,49,6,96,0.484211,0.884615,24.873096,0.482234
2,test,scope_risk_label,197,37,114,34,80,3,80,0.298246,0.918919,40.609137,0.578680
3,validation,dependency_risk_label,198,16,84,10,74,6,108,0.119048,0.625000,37.373737,0.424242
4,validation,schedule_risk_label,198,52,107,37,70,15,76,0.345794,0.711538,35.353535,0.540404
5,validation,scope_risk_label,198,33,65,22,43,11,122,0.338462,0.666667,21.717172,0.328283


Actual lead-time days: NOT ESTABLISHED — need independent event-onset timestamps and a documented episode definition.


## 5. Alert episodes and first warning checkpoints (descriptive only)
We group consecutive positive-label checkpoints within each instance and risk. This is a **proxy episode**, not a verified real-world failure episode. The first predicted warning is not a validated lead-time estimate.

In [15]:
episodes=[]
for (part,risk,inst),g in joined.groupby(['partition','risk','instance_id']):
    g=g.sort_values(['checkpoint_time','week_number'])
    active=g[g['actual_label'].eq(1)].copy()
    if active.empty: continue
    active['episode_id']=active['week_number'].diff().ne(1).cumsum()
    for ep,h in active.groupby('episode_id'):
        detected=h[h['predicted_label'].eq(1)]
        episodes.append({'partition':part,'risk':risk,'instance_id':inst,'episode':int(ep),'first_positive_checkpoint':h['checkpoint_time'].min(),'last_positive_checkpoint':h['checkpoint_time'].max(),'positive_checkpoints':len(h),'detected_within_positive_checkpoints':not detected.empty,'first_detected_checkpoint':detected['checkpoint_time'].min() if not detected.empty else pd.NaT})
episodes=pd.DataFrame(episodes)
episodes.to_csv(OUT/'proxy_positive_episodes.csv',index=False)
if len(episodes): display(episodes.groupby(['partition','risk'])['detected_within_positive_checkpoints'].agg(['size','sum','mean']))
print('Proxy episode detection is descriptive and cannot substitute for outcome-date lead time.')

size  sum      mean
partition  risk                                      
test       dependency_risk_label     5    4  0.800000
           schedule_risk_label      12   11  0.916667
           scope_risk_label         11    9  0.818182
validation dependency_risk_label    11    7  0.636364
           schedule_risk_label      17   13  0.764706
           scope_risk_label         14   12  0.857143

Proxy episode detection is descriptive and cannot substitute for outcome-date lead time.


## 6. Leakage-risk register and acceptance decision
Explicitly separate checks performed from checks still needed. A clean audit is not a certificate of zero leakage.

In [16]:
register=[
 {'check':'Instance-disjoint train/validation/test','status':'PASS' if split['instance_id'].is_unique and joined['evaluation_partition'].eq(joined['partition']).all() else 'FAIL','interpretation':'Split keys are disjoint; project-level independence requires project ID check.'},
 {'check':'Saved predictions match 14-day labels','status':'PASS','interpretation':'Labels match exported feature dataset; label construct validity is separate.'},
 {'check':'Checkpoint chronological order','status':'PASS' if chron[['duplicate_weeks','non_increasing_checkpoint_times','non_increasing_week_numbers']].sum().sum()==0 else 'FLAG','interpretation':'Metadata chronology only.'},
 {'check':'Selected feature names','status':'PASS' if len(scan) and not scan['suspicious_name'].any() and scan['present_in_dataset'].all() else 'REVIEW','interpretation':'Name screen only, not feature lineage.'},
 {'check':'Historical feature availability','status':'NOT_ESTABLISHED','interpretation':'Requires recomputing selected features from strictly as-of source snapshots.'},
 {'check':'Outcome source timestamps and event eligibility','status':'NOT_ESTABLISHED','interpretation':'Requires independent Jira event reconstruction and 14-day horizon alignment.'},
 {'check':'Exact early-warning lead time','status':'NOT_ESTABLISHED','interpretation':'Requires dated outcome onset and a defined event/episode mapping.'},
 {'check':'Untouched holdout test','status':'NOT_ESTABLISHED','interpretation':'Exploratory test was previously examined; do not claim untouched test generalization.'},
]
reg=pd.DataFrame(register);reg.to_csv(OUT/'leakage_risk_register.csv',index=False);display(reg)
summary={'checkpoint_count':int(len(features)),'instance_count':int(features['instance_id'].nunique()),'prediction_rows':int(len(joined)),'selected_feature_count':len(feature_names),'chronology_violations':int(chron[['duplicate_weeks','non_increasing_checkpoint_times','non_increasing_week_numbers']].sum().sum()),'feature_name_flags':int(scan['suspicious_name'].sum()) if len(scan) else None,'leakage_proven_absent':False,'exact_lead_time_days':'NOT_ESTABLISHED','note':'Checkpoint-level 14-day warning metrics are descriptive. Exploratory test is not an untouched holdout.'}
(OUT/'temporal_leakage_early_warning_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2));print('Saved:',OUT)

,check,status,interpretation
0,Instance-disjoint train/validation/test,PASS,Split keys are disjoint; project-level indepen...
1,Saved predictions match 14-day labels,PASS,Labels match exported feature dataset; label c...
2,Checkpoint chronological order,PASS,Metadata chronology only.
3,Selected feature names,PASS,"Name screen only, not feature lineage."
4,Historical feature availability,NOT_ESTABLISHED,Requires recomputing selected features from st...
5,Outcome source timestamps and event eligibility,NOT_ESTABLISHED,Requires independent Jira event reconstruction...
6,Exact early-warning lead time,NOT_ESTABLISHED,Requires dated outcome onset and a defined eve...
7,Untouched holdout test,NOT_ESTABLISHED,Exploratory test was previously examined; do n...


{
  "checkpoint_count": 987,
  "instance_count": 30,
  "prediction_rows": 1185,
  "selected_feature_count": 34,
  "chronology_violations": 0,
  "feature_name_flags": 0,
  "leakage_proven_absent": false,
  "exact_lead_time_days": "NOT_ESTABLISHED",
  "note": "Checkpoint-level 14-day warning metrics are descriptive. Exploratory test is not an untouched holdout."
}
Saved: /content/drive/MyDrive/02_Risk_Prediction/Results/Temporal_Leakage_Early_Warning_V1


Phase 2 is technically functioning, but the evaluation shows a meaningful false-alarm problem and some unresolved historical-validity questions. These are important findings to report, not reasons to discard the project.